# Florence-2 Layout-Aware OCR Benchmark (Micromamba Environment)
**Pierce 1890 Medical Adviser · Team G07 · A2 OCR benchmarking**

This notebook downloads **Micromamba** and creates an isolated Conda environment in `/kaggle/working/mamba_env`.
This isolates all Python & C++ libraries from Kaggle system packages with zero dependency conflicts.

## Kaggle Settings
- Accelerator: **GPU T4 x2**
- Datasets attached:
  1. `kmazd1110/dl-peoples-common-sense-med-advisor` (PDF)
  2. `cruelangelssprint/pierce-1890-figure-and-ocr-outputs` (Chandra `chunks.jsonl`)
  3. `kmazd1110/pierce-book-gt` (`labels.jsonl`)

## Cell 1 — Download Micromamba & create isolated environment (`/kaggle/working/mamba_env`)

In [ ]:
import os, subprocess, sys
from pathlib import Path

MAMBA_BIN = Path("/tmp/bin/micromamba")
ENV_DIR   = Path("/kaggle/working/mamba_env")
PY_BIN    = ENV_DIR / "bin" / "python"
PIP_BIN   = ENV_DIR / "bin" / "pip"

# Step 1: Download micromamba binary to /tmp/bin/micromamba if not present
if not MAMBA_BIN.exists():
    print("Downloading Micromamba binary...")
    Path("/tmp/bin").mkdir(parents=True, exist_ok=True)
    subprocess.run(
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp bin/micromamba",
        shell=True, check=True
    )
    print(f"Micromamba installed to {MAMBA_BIN}")

# Step 2: Create isolated Micromamba environment
if not PY_BIN.exists():
    print(f"Creating Micromamba environment at {ENV_DIR}...")
    subprocess.run([
        str(MAMBA_BIN), "create", "-y", "-p", str(ENV_DIR),
        "-c", "conda-forge", "python=3.10", "pip"
    ], check=True)
    print("Micromamba base env created. Installing PyTorch & OCR packages...")
    subprocess.run([
        str(PIP_BIN), "install", "-q",
        "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121"
    ], check=True)
    subprocess.run([
        str(PIP_BIN), "install", "-q",
        "transformers==4.41.2",
        "timm==0.9.16",
        "einops",
        "pillow==10.3.0",
        "pymupdf",
        "jiwer",
        "opencv-python-headless",
    ], check=True)
    print("All packages installed successfully in Micromamba environment!")
else:
    print(f"Micromamba environment already exists at {ENV_DIR}")

# Verify environment
subprocess.run([str(PY_BIN), "-c", "import torch, transformers, timm; print('Torch CUDA:', torch.cuda.is_available()); print('Transformers:', transformers.__version__); print('Timm:', timm.__version__)"])

## Cell 2 — Verify dataset paths

In [ ]:
PDF = Path("/kaggle/input/datasets/kmazd1110/dl-peoples-common-sense-med-advisor/EN_The-Peoples-Common-Sense-Medical-Adviser.pdf")
CHANDRA = Path("/kaggle/input/datasets/cruelangelssprint/pierce-1890-figure-and-ocr-outputs/chandra/chunks.jsonl")
GT = Path("/kaggle/input/datasets/kmazd1110/pierce-book-gt/labels.jsonl")

for p in [PDF, CHANDRA, GT]:
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {p}")
    assert p.exists(), f"Missing: {p}"

## Cell 3 — Execute Florence-2 OCR Benchmark in isolated environment

In [ ]:
# Write the complete Florence-2 benchmark script
script_path = Path("/kaggle/working/florence2_layout_bench.py")
script_path.write_text('#!/usr/bin/env python3\n"""\nFlorence-2 Layout-Aware OCR Benchmark (Isolated Environment Script)\nPierce 1890 Medical Adviser · Team G07 · A2 OCR benchmarking\n"""\n\nimport csv\nimport json\nimport os\nimport re\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nimport cv2\nimport fitz  # PyMuPDF\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom jiwer import cer as compute_cer, wer as compute_wer\nfrom transformers import AutoModelForCausalLM, AutoProcessor\nfrom transformers.dynamic_module_utils import get_imports\n\n# ── Paths ─────────────────────────────────────────────────────────────────────\nPDF_PATH = Path(\n    "/kaggle/input/datasets/kmazd1110/dl-peoples-common-sense-med-advisor"\n    "/EN_The-Peoples-Common-Sense-Medical-Adviser.pdf"\n)\nCHANDRA_PATH = Path(\n    "/kaggle/input/datasets/cruelangelssprint/pierce-1890-figure-and-ocr-outputs"\n    "/chandra/chunks.jsonl"\n)\nLABELS_PATH = Path("/kaggle/input/datasets/kmazd1110/pierce-book-gt/labels.jsonl")\n\nOUT_DIR = Path("/kaggle/working/florence2_layout_bench")\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nMODEL_ID = "microsoft/Florence-2-base"\nOCR_TASK = "<OCR>"\nMAX_TOKENS = 512\nDPI = 300\n\nDEVICE = "cuda" if torch.cuda.is_available() else "cpu"\nprint(f"Device: {DEVICE}")\n\n# ── Bypass optional flash_attn import requirement ──────────────────────────────\ndef fixed_get_imports(filename: str | os.PathLike) -> list[str]:\n    imports = get_imports(filename)\n    if "flash_attn" in imports:\n        imports.remove("flash_attn")\n    return imports\n\nprint(f"Loading {MODEL_ID} on {DEVICE}...")\nt0 = time.time()\nprocessor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)\n\nwith patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):\n    model = AutoModelForCausalLM.from_pretrained(\n        MODEL_ID,\n        trust_remote_code=True,\n        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,\n    ).to(DEVICE)\n\nmodel.eval()\nprint(f"Model loaded in {time.time()-t0:.1f}s | dtype={next(model.parameters()).dtype}")\n\n# ── Helpers ───────────────────────────────────────────────────────────────────\ndef _chandra_label_kind(label) -> str:\n    TEXT_LABELS = {\n        "text", "section-header", "caption",\n        "footnote", "list-group", "table",\n    }\n    if label is None:\n        return "skip"\n    return "text" if str(label).lower().strip() in TEXT_LABELS else "skip"\n\ndef _strip_html(html: str) -> str:\n    return re.sub(r"<[^>]+>", " ", html).strip()\n\ndef render_page(doc: fitz.Document, page_idx: int, dpi: int = DPI) -> np.ndarray:\n    pix = doc[page_idx].get_pixmap(dpi=dpi)\n    arr = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)\n    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR if pix.n == 3 else cv2.COLOR_RGBA2BGR)\n\ndef bbox_to_pixel(bbox: list, page_box: list, img_w: int, img_h: int) -> tuple[int, int, int, int] | None:\n    pb_x0, pb_y0, pb_x1, pb_y1 = (float(v) for v in page_box)\n    cw = pb_x1 - pb_x0\n    ch = pb_y1 - pb_y0\n    if cw <= 0.0 or ch <= 0.0:\n        return None\n    x0, y0, x1, y1 = (float(v) for v in bbox)\n    px0 = max(0, int((x0 - pb_x0) / cw * img_w))\n    py0 = max(0, int((y0 - pb_y0) / ch * img_h))\n    px1 = min(img_w, int((x1 - pb_x0) / cw * img_w))\n    py1 = min(img_h, int((y1 - pb_y0) / ch * img_h))\n    return px0, py0, px1, py1\n\n@torch.inference_mode()\ndef florence_ocr_crop(img_bgr: np.ndarray, x0: int, y0: int, x1: int, y1: int) -> str:\n    if x1 <= x0 or y1 <= y0:\n        return ""\n    crop_bgr = img_bgr[y0:y1, x0:x1]\n    if crop_bgr.size == 0:\n        return ""\n    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)\n    pil_img = Image.fromarray(crop_rgb)\n\n    inputs = processor(\n        text=OCR_TASK,\n        images=pil_img,\n        return_tensors="pt",\n    ).to(DEVICE, dtype=torch.float16 if DEVICE == "cuda" else torch.float32)\n\n    generated_ids = model.generate(\n        input_ids=inputs["input_ids"],\n        pixel_values=inputs["pixel_values"],\n        max_new_tokens=MAX_TOKENS,\n        num_beams=3,\n        do_sample=False,\n    )\n    raw = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]\n    parsed = processor.post_process_generation(\n        raw, task=OCR_TASK, image_size=(pil_img.width, pil_img.height)\n    )\n    return str(parsed.get(OCR_TASK, "")).strip()\n\ndef normalize(text: str) -> str:\n    return re.sub(r"\\s+", " ", text).strip()\n\n# ── Main processing ───────────────────────────────────────────────────────────\ndef main():\n    print("Loading Chandra layout blocks...")\n    chandra_blocks: dict[str, list[dict]] = defaultdict(list)\n    skipped_label = 0\n    with CHANDRA_PATH.open(encoding="utf-8") as f:\n        for line in f:\n            if not line.strip():\n                continue\n            row = json.loads(line)\n            book_page = row.get("book_page")\n            if book_page is None:\n                continue\n            label = row.get("label", "")\n            if _chandra_label_kind(label) == "skip":\n                skipped_label += 1\n                continue\n            page_id = f"p{int(book_page):04d}"\n            chandra_blocks[page_id].append({\n                "page_box": row.get("page_box"),\n                "bbox": row.get("bbox"),\n                "label": label,\n                "content": _strip_html(row.get("content", "")),\n            })\n    print(f"Loaded {sum(len(v) for v in chandra_blocks.values())} text blocks across {len(chandra_blocks)} pages.")\n\n    print("Loading GT labels...")\n    gt_labels = {}\n    with LABELS_PATH.open(encoding="utf-8") as f:\n        for line in f:\n            if not line.strip():\n                continue\n            row = json.loads(line)\n            gt_labels[row["page_id"]] = row["text"]\n\n    doc = fitz.open(str(PDF_PATH))\n    all_page_ids = sorted(chandra_blocks.keys())\n    results = []\n    t_total = time.time()\n\n    transcripts_path = OUT_DIR / "page_transcripts.jsonl"\n    out_f = transcripts_path.open("w", encoding="utf-8")\n\n    for page_num, page_id in enumerate(all_page_ids):\n        pdf_idx = int(page_id[1:]) - 1\n        if pdf_idx < 0 or pdf_idx >= doc.page_count:\n            continue\n        t0 = time.time()\n        img = render_page(doc, pdf_idx, DPI)\n        img_h, img_w = img.shape[:2]\n\n        block_texts = []\n        for blk in chandra_blocks[page_id]:\n            px_res = bbox_to_pixel(blk.get("bbox"), blk.get("page_box"), img_w, img_h)\n            if px_res is None:\n                continue\n            px0, py0, px1, py1 = px_res\n            text = florence_ocr_crop(img, px0, py0, px1, py1)\n            if text:\n                block_texts.append(text)\n\n        page_text = "\\n".join(block_texts)\n        elapsed = time.time() - t0\n\n        row = {"page_id": page_id, "text": page_text, "n_blocks": len(block_texts), "elapsed_s": round(elapsed, 2)}\n        results.append(row)\n        out_f.write(json.dumps(row, ensure_ascii=False) + "\\n")\n\n        if page_num % 50 == 0 or page_num < 3:\n            print(f"  [{page_num+1:4d}/{len(all_page_ids)}] {page_id} — {len(block_texts)} blocks, {elapsed:.1f}s")\n\n    out_f.close()\n    print(f"OCR finished in {(time.time()-t_total)/60:.1f} min.")\n\n    # ── Scoring ───────────────────────────────────────────────────────────────\n    transcript_map = {r["page_id"]: r["text"] for r in results}\n    scored_pages = []\n    for page_id, ref_text in sorted(gt_labels.items()):\n        hyp_text = transcript_map.get(page_id, "")\n        ref_norm = normalize(ref_text)\n        hyp_norm = normalize(hyp_text)\n        if not ref_norm:\n            continue\n        c = compute_cer(ref_norm, hyp_norm)\n        w = compute_wer(ref_norm, hyp_norm)\n        ref_words = set(ref_norm.lower().split())\n        hyp_words = set(hyp_norm.lower().split())\n        tp = len(ref_words & hyp_words)\n        prec = tp / len(hyp_words) if hyp_words else 0.0\n        rec = tp / len(ref_words) if ref_words else 0.0\n        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0\n        scored_pages.append({"page_id": page_id, "cer": round(c, 4), "wer": round(w, 4), "word_f1": round(f1, 4)})\n\n    mean_cer = float(np.mean([p["cer"] for p in scored_pages]))\n    mean_wer = float(np.mean([p["wer"] for p in scored_pages]))\n    mean_f1 = float(np.mean([p["word_f1"] for p in scored_pages]))\n\n    score_path = OUT_DIR / "heldout_scores.csv"\n    with score_path.open("w", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=list(scored_pages[0]))\n        writer.writeheader()\n        writer.writerows(scored_pages)\n\n    lines = [\n        "# Florence-2 Layout-Aware OCR Benchmark",\n        f"- Mean CER    : {mean_cer:.4f}",\n        f"- Mean WER    : {mean_wer:.4f}",\n        f"- Mean Word F1: {mean_f1:.4f}",\n    ]\n    report_path = OUT_DIR / "report.md"\n    report_path.write_text("\\n".join(lines) + "\\n")\n    print(f"Results saved. Mean CER: {mean_cer:.4f}, Mean WER: {mean_wer:.4f}")\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
print(f"Script written to {script_path}")

print("Launching Florence-2 layout benchmark via Micromamba environment...")
import subprocess
p = subprocess.run(["/kaggle/working/mamba_env/bin/python", str(script_path)])
assert p.returncode == 0, f"Benchmark failed with exit code {p.returncode}"

## Cell 4 — Display Summary Report

In [ ]:
report_file = Path("/kaggle/working/florence2_layout_bench/report.md")
if report_file.exists():
    print(report_file.read_text())
else:
    print("Report not generated yet.")